# So sanh kernel_initializer trong tf.keras: zeros / random lon / He

Ban minh hoa dung **TensorFlow / Keras** song song voi bai 03 (tu viet `initialize_parameters_zeros`,
`initialize_parameters_random`, `initialize_parameters_he` bang NumPy). O `tf.keras`, cach khoi tao
trong so la mot tham so co san cua moi lop `Dense`: `kernel_initializer`. Ban se thay 3 cach khoi
tao da tu cai tay o bai 03 co san **dung ten** trong Keras (`Zeros`, `RandomNormal`, `HeNormal`).

In [ ]:
# --- Google Colab setup: bo qua tu dong neu chay Jupyter local ---
import sys, os

if "google.colab" in sys.modules:
    REPO_DIR = "/content/AI-Teaching-Labs"
    if not os.path.isdir(REPO_DIR):
        !git clone --depth 1 https://github.com/NonomiyaIzumi/AI-Teaching-Labs.git {REPO_DIR}
    NOTEBOOK_DIR = f"{REPO_DIR}/Deep Learning/Module 1/keras-tensorflow/03_Vanishing-Exploding-gradient-va-Khoi-tao-trong-so"
    os.chdir(NOTEBOOK_DIR)
    print("Da chuyen working directory sang:", os.getcwd())
else:
    print("Khong phai Colab - dung working directory hien tai:", os.getcwd())
    print("Neu chay Jupyter local: tu thu muc keras-tensorflow/, chay 'uv sync' truoc.")

## 1. Du lieu & kien truc

Dung lai bo du lieu **Pima Indians Diabetes** va kien truc `[8, 10, 5, 1]` giong het bai 03 (NumPy,
`layers_dims = [X.shape[0], 10, 5, 1]`) de ket qua so sanh duoc cong bang.

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers

from keras_utils import load_dataset, compile_and_train, evaluate_classification

tf.config.experimental.enable_op_determinism()  # dam bao ket qua giong het nhau giua cac lan chay

train_X, train_Y, test_X, test_Y = load_dataset()
print("train_X:", train_X.shape, " train_Y:", train_Y.shape)
print("test_X: ", test_X.shape, " test_Y: ", test_Y.shape)

## 2. Kien truc dung chung - Exercise 1

Ca 3 lan train chi khac nhau o **initializer** truyen vao - moi lop `Dense` deu dung chung mot
`kernel_initializer`, giong cach `initialize_parameters_zeros/random/he` o bai 03 deu ap dung dong
nhat cho moi layer.

### Exercise 1 - `build_model_with_initializer`

In [ ]:
def build_model_with_initializer(input_dim, initializer):
    '''
    Xay dung kien truc [input_dim, 10, 5, 1] (Dense -> ReLU -> Dense -> ReLU -> Dense -> Sigmoid),
    ap dung CUNG MOT initializer cho ca 3 lop Dense.

    Arguments:
    input_dim   -- so dac trung dau vao (8 cho Pima Diabetes)
    initializer -- mot tf.keras.initializers.Initializer (Zeros / RandomNormal / HeNormal ...)

    Returns: mot tf.keras.Model CHUA compile.
    '''
    # (~6 dong) dung keras.Sequential + layers.Dense(..., kernel_initializer=initializer)
    # model = keras.Sequential([
    #     keras.Input(shape=(input_dim,)),
    #     layers.Dense(10, activation="relu", kernel_initializer=initializer),
    #     layers.Dense(5, activation="relu", kernel_initializer=initializer),
    #     layers.Dense(1, activation="sigmoid", kernel_initializer=initializer),
    # ])
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE
    return model

In [ ]:
#@title Answer { display-mode: "form" }
def build_model_with_initializer(input_dim, initializer):
    '''
    Xay dung kien truc [input_dim, 10, 5, 1] (Dense -> ReLU -> Dense -> ReLU -> Dense -> Sigmoid),
    ap dung CUNG MOT initializer cho ca 3 lop Dense.

    Arguments:
    input_dim   -- so dac trung dau vao (8 cho Pima Diabetes)
    initializer -- mot tf.keras.initializers.Initializer (Zeros / RandomNormal / HeNormal ...)

    Returns: mot tf.keras.Model CHUA compile.
    '''
    # (~6 dong) dung keras.Sequential + layers.Dense(..., kernel_initializer=initializer)
    # YOUR CODE STARTS HERE
    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),
        layers.Dense(10, activation="relu", kernel_initializer=initializer),
        layers.Dense(5, activation="relu", kernel_initializer=initializer),
        layers.Dense(1, activation="sigmoid", kernel_initializer=initializer),
    ])
    # YOUR CODE ENDS HERE
    return model

## 3. Huan luyen & so sanh 3 cach khoi tao

- `"zeros"` -> `keras.initializers.Zeros()` (tuong duong `initialize_parameters_zeros`)
- `"random_bad"` -> `keras.initializers.RandomNormal(stddev=10.0)` (tuong duong `randn(...) * 10`
  trong `initialize_parameters_random`)
- `"he"` -> `keras.initializers.HeNormal()` (tuong duong `randn(...) * sqrt(2/fan_in)` trong
  `initialize_parameters_he`)

Dung cung `learning_rate=0.01` (giong `model()` mac dinh o bai 03), full-batch gradient descent.

In [ ]:
initializers = {
    "zeros":      keras.initializers.Zeros(),
    "random_bad": keras.initializers.RandomNormal(mean=0.0, stddev=10.0, seed=3),
    "he":         keras.initializers.HeNormal(seed=3),
}

EPOCHS = 2000
histories = {}
for name, init in initializers.items():
    tf.random.set_seed(3)
    model = build_model_with_initializer(train_X.shape[0], init)
    history = compile_and_train(
        model, train_X, train_Y,
        optimizer=keras.optimizers.SGD(learning_rate=0.01),
        epochs=EPOCHS, batch_size=None, verbose=0,
    )
    histories[name] = {"model": model, "history": history}
    final_cost = history.history["loss"][-1]
    print(f"{name:10s} -> cost cuoi cung (train) = {final_cost:.4f}")

In [ ]:
plt.figure(figsize=(7, 4.5))
for name in initializers:
    plt.plot(histories[name]["history"].history["loss"], label=name)
plt.xlabel("epoch")
plt.ylabel("cost")
plt.title("Cost qua qua trinh train - so sanh kernel_initializer")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
for name in initializers:
    evaluate_classification(histories[name]["model"], test_X, test_Y, f'kernel_initializer = "{name}" (test set)')

## 4. Tong ket

- Sau 2000 epoch (`learning_rate=0.01`), ket qua tren test set: **`zeros`** 64.9% accuracy nhung
  **Precision = Recall = 0** (khong phat hien duoc ca nao mac benh - hoan toan khong hoc duoc gi,
  giong het bai 03 NumPy); **`random_bad`** (stddev=10) cung 64.9%/Recall=0 - cost co giam nhe
  (0.5413) nhung van khong tach duoc lop nao; **`he`** dat 68.2% accuracy, Recall **51.9%** - hoc
  duoc mot tin hieu du doan that su. Ca 3 con so nay se tot hon nua neu tang so epoch (ban NumPy o
  bai 03 dung toi 15000 iteration, chi thu 2000 epoch de notebook chay nhanh khi minh hoa).
- Nguyen nhan dinh tinh giong het bai 03: **`zeros`** khien moi neuron trong cung mot lop hoc y het
  nhau (symmetry chua bao gio bi pha vo); **`random` voi stddev qua lon** day logit ve vung bao hoa
  cua Sigmoid/ReLU ngay tu dau, gay vanishing/exploding gradient; **`he`** giu phuong sai on dinh
  qua cac lop nen hoc tot hon han 2 cach con lai.
- Diem khac biet lon nhat so voi bai 03: ban **khong can tu viet cong thuc** `np.random.randn(...) *
  sqrt(2./layers_dims[l-1])` - chi can goi dung ten `HeNormal()`. Rui ro o day chuyen tu "viet sai
  cong thuc" (bai 03) sang "chon sai initializer cho kien truc/activation dang dung" (vd `HeNormal`
  hop voi ReLU hon la voi Tanh, noi `GlorotNormal`/Xavier thuong phu hop hon).
- Ky thuat **gradient flow visualization** (bai 03) van ap dung duoc voi Keras: co the doc
  `tape.gradient(...)` cho tung bien trong `model.trainable_variables` moi vai epoch (tuong tu
  `model_with_grad_tracking` o bai 03) neu muon ve lai bieu do `||dW||` theo layer.